# 🎬 CogVideoX-5B-I2V otimizado para Colab

Este notebook gera vídeos a partir de uma **imagem** + **prompt textual**, usando o modelo **CogVideoX-5B-I2V** com ajustes para rodar em Colab com GPU T4.

✅ Geração com 12 frames
✅ Inferência em `fp16`
✅ Offload de memória ativado


In [ ]:
# ✅ Instalar dependências (ajustadas para compatibilidade e leveza)
!pip install diffusers==0.33.1 transformers accelerate einops gradio ffmpeg-python safetensors --quiet

In [ ]:
# 🔑 Login Hugging Face (necessário para carregar o modelo)
from huggingface_hub import login
login()

In [ ]:
# 📦 Imports e configurações
import os
import torch
from PIL import Image
import gradio as gr
from diffusers import (
    CogVideoXImageToVideoPipeline,
    AutoencoderKLCogVideoX,
    CogVideoXTransformer3DModel,
)
from diffusers.utils import export_to_video, load_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("outputs", exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# 🚀 Carregar modelo com otimizadores de memória
transformer = CogVideoXTransformer3DModel.from_pretrained(
    "THUDM/CogVideoX-5b-I2V", subfolder="transformer", torch_dtype=torch.float16
)
text_encoder = AutoencoderKLCogVideoX.from_pretrained(
    "THUDM/CogVideoX-5b-I2V", subfolder="vae", torch_dtype=torch.float16
)
pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    text_encoder=text_encoder,
    transformer=transformer,
    vae=text_encoder,
    torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()
pipe.to(device)

In [ ]:
# 🎥 Função de geração com num_frames reduzido (12)
def generate_video(image_path, prompt):
    image = load_image(image_path).convert("RGB").resize((720, 480))
    result = pipe(
        image=image,
        prompt=prompt,
        guidance_scale=5,
        num_inference_steps=20,
        num_frames=12
    )
    frames = result.frames[0]
    output_path = "outputs/cogvideo_output.mp4"
    export_to_video(frames, output_path, fps=8)
    return output_path

In [ ]:
# 🎛️ Interface Gradio
with gr.Blocks() as demo:
    gr.Markdown("## 🎬 CogVideoX: Geração de vídeo otimizada para Colab")
    with gr.Row():
        img_input = gr.Image(type="filepath", label="Imagem de entrada")
        prompt_input = gr.Textbox(label="Prompt (ex: 'a futuristic city at night')")
    generate_btn = gr.Button("Gerar Vídeo")
    output_video = gr.Video(label="🎞️ Resultado")
    generate_btn.click(fn=generate_video, inputs=[img_input, prompt_input], outputs=output_video)
    demo.launch(share=True)